# 02 · Cleaning

In [1]:
import pandas as pd

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

In [3]:
df = pd.read_csv("../data/raw/jabodetabek_house_price.csv")

In [4]:
df_clean = df.copy()

df_clean["city"] = df_clean["city"].str.strip()

In [5]:
komersial = r"\b(?:kost|kos|kostan|kosan)\b|hotel|guest ?house|penginapan|ex kantor|ex salon|ex restoran|lelang"

hasil = df[df["title"].str.contains(komersial, case=False, na=False)]
print(len(hasil))
hasil[["title", "bedrooms", "city", "bathrooms", "price_in_rp"]]

28


,title,bedrooms,city,bathrooms,price_in_rp
38,"Lelang Perumahan 2 Lt Metland Bekasi, Jawa Barat",3.0,Bekasi,2.0,7.000000e+08
346,Di Jual Cepat Rumah Kost kostan di Rawalumbu Bekasi,36.0,Bekasi,36.0,9.000000e+09
1333,"Rumah Kos di Dramaga, Dekat Kampus Ipb, Bogor",37.0,Bogor,15.0,2.700000e+09
1371,Jual Villa 4600 M Lokasi Cipanas Depan Hotel Aston Surat SHM,4.0,Bogor,3.0,7.000000e+09
1372,"Jual Villa Ex Restoran Luas Tanah 1500 Meter Surat SHM Harga 2,5m",3.0,Bogor,3.0,2.500000e+09
1392,Hotel di Kawasan Pariwisata Dan Masih Aktif di Gadog Ciawi Bogor,13.0,Bogor,8.0,4.500000e+10
1709,Rumah ex kantor di Limo,2.0,Depok,1.0,2.950000e+08
2007,Chandra Rumah Kos Baru Uk 8x17m Lokasi Bagus di Jelambar,10.0,Jakarta Barat,11.0,5.500000e+09
2030,Dhyana Rumah Kos Baru Uk 8x17m Lokasi Bagus di Jelambar,10.0,Jakarta Barat,11.0,5.500000e+09
2070,"Lelang Rumah Mewah 2 Lt, di Jagakarsa, Jakarta Selatan",4.0,Jakarta Selatan,3.0,1.200000e+10


In [6]:
dilindungi = [3127, 1709, 3446]

mask = df_clean["title"].str.contains(komersial, case=False, na=False) & ~df_clean.index.isin(dilindungi)
print(mask.sum())

25


In [7]:
df_clean = df_clean[~mask]

In [8]:
kw_bukan_harga_pasar = r"oper kredit|over kredit|overkredit|take over|oper murah|lelang"
mask = df_clean["title"].str.contains(kw_bukan_harga_pasar, case=False, na=False)
print(mask.sum())
df_clean = df_clean[~mask]

10


In [9]:
# luas bangunan kosong
df_clean = df_clean[df_clean["building_size_m2"].notna()]

# luas bangunan < 10 m2
df_clean = df_clean[df_clean["building_size_m2"] >= 10]

# bangunan kecil tapi harga/m2 tidak wajar
m = (df_clean["building_size_m2"] < 30) & (df_clean["price_in_rp"] / df_clean["building_size_m2"] > 100e6)
print(list(df_clean[m].index))          # harus 3060 dan 3333
df_clean = df_clean[~m]

# kamar tidur >= 10 dan kamar mandi >= kamar tidur (pola kos)
m = (df_clean["bedrooms"] >= 10) & (df_clean["bathrooms"] >= df_clean["bedrooms"])
print(list(df_clean[m].index))          # harus 1370, 2305, 2355, 2357, 2642, 2659
df_clean = df_clean[~m]

# keputusan manual per baris
manual = [2906, 21, 269, 1455, 1671, 1008, 1010]
df_clean = df_clean.drop(index=manual, errors="ignore")

[3060, 3333]
[1370, 2305, 2355, 2357, 2642, 2659]


Pola "kamar tidur ≥10 dan kamar mandi ≥ kamar tidur" adalah ciri kos yang lolos dari filter judul. Baris manual (2906, 21, 269, 1455) sudah dicek satu per satu di notebook 01.

In [10]:
df_clean = df_clean.drop(columns=["lat", "long", "year_built"])

In [11]:
print("Terbuang:", len(df) - len(df_clean), "| sisa:", len(df_clean))
df_clean[["price_in_rp", "land_size_m2", "building_size_m2", "bedrooms", "bathrooms"]].describe().T

Terbuang: 53 | sisa: 3500


,count,mean,std,min,25%,50%,75%,max
price_in_rp,3500.0,3.920117e+09,9.640743e+09,100000000.0,800000000.0,1.500000e+09,3.500000e+09,2.500000e+11
land_size_m2,3500.0,1.956511e+02,3.661181e+02,12.0,75.0,1.060000e+02,1.832500e+02,8.000000e+03
building_size_m2,3500.0,1.803097e+02,2.160883e+02,19.0,66.0,1.100000e+02,2.000000e+02,5.000000e+03
bedrooms,3468.0,3.190311e+00,1.100174e+00,1.0,2.0,3.000000e+00,4.000000e+00,1.600000e+01
bathrooms,3473.0,2.492370e+00,1.211034e+00,1.0,2.0,2.000000e+00,3.000000e+00,9.000000e+00


In [12]:
df_clean.shape

(3500, 24)

In [13]:
tukar = df_clean["furnishing"] == "baru"
print(tukar.sum())   
df_clean[tukar][["title", "property_condition", "furnishing"]] 

df_clean.loc[tukar, ["property_condition", "furnishing"]] = df_clean.loc[tukar, ["furnishing", "property_condition"]].values

29


In [14]:
print(df_clean["furnishing"].value_counts(dropna=False))
print(df_clean["property_condition"].value_counts(dropna=False))

furnishing
unfurnished       2071
semi furnished     824
NaN                378
furnished          227
Name: count, dtype: int64
property_condition
bagus             1411
baru              1347
bagus sekali       252
NaN                241
sudah renovasi     156
butuh renovasi      93
Name: count, dtype: int64


In [15]:
kolom_cek = [c for c in df_clean.columns if c not in ["url", "ads_id", "facilities", "title", "address"]]

df_clean = df_clean.drop_duplicates(subset=kolom_cek, keep="first")

In [16]:
print("Terbuang:", len(df) - len(df_clean), "| sisa:", len(df_clean))

Terbuang: 382 | sisa: 3171


In [17]:
import os
os.makedirs("../data/processed", exist_ok=True)
df_clean.to_csv("../data/processed/rumah_clean.csv", index=False)